In [9]:
#!/usr/bin/env python
# coding: utf-8

# In[11]:


# -*- coding: utf-8 -*-
'''ResNet50 model for Keras, with manual definition.

Reference:
- Deep Residual Learning for Image Recognition: https://arxiv.org/abs/1512.03385
'''

from __future__ import print_function
import numpy as np
import warnings
from tensorflow.keras.layers import (
    Input, Dense, Activation, Flatten, Conv2D, MaxPooling2D, GlobalMaxPooling2D,
    ZeroPadding2D, AveragePooling2D, GlobalAveragePooling2D, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
import tensorflow.keras.backend as K
from tensorflow.keras.utils import get_file
from tensorflow.keras.applications.imagenet_utils import (
    decode_predictions, preprocess_input
)
from tensorflow.keras.utils import plot_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers

import os

# Weights URLh5
WEIGHTS_PATH = 'https://github.com/fchollet/deep-learning-models/releases/download/v0.2/resnet50_weights_tf_dim_ordering_tf_kernels.'
WEIGHTS_PATH_NO_TOP = 'https://github.com/fchollet/deep-learning-models/releases/download/v0.2/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5'

def identity_block(input_tensor, kernel_size, filters, stage, block):
    filters1, filters2, filters3 = filters
    bn_axis = 3 if K.image_data_format() == 'channels_last' else 1
    conv_base = f'res{stage}{block}_branch'
    bn_base = f'bn{stage}{block}_branch'

    x = Conv2D(filters1, (1, 1), name=conv_base + '2a')(input_tensor)
    x = BatchNormalization(axis=bn_axis, name=bn_base + '2a')(x)
    x = Activation('relu')(x)

    x = Conv2D(filters2, kernel_size, padding='same', name=conv_base + '2b')(x)
    x = BatchNormalization(axis=bn_axis, name=bn_base + '2b')(x)
    x = Activation('relu')(x)

    x = Conv2D(filters3, (1, 1), name=conv_base + '2c')(x)
    x = BatchNormalization(axis=bn_axis, name=bn_base + '2c')(x)

    x = layers.add([x, input_tensor])
    x = Activation('relu')(x)
    return x

def conv_block(input_tensor, kernel_size, filters, stage, block, strides=(2, 2)):
    filters1, filters2, filters3 = filters
    bn_axis = 3 if K.image_data_format() == 'channels_last' else 1
    conv_base = f'res{stage}{block}_branch'
    bn_base = f'bn{stage}{block}_branch'

    x = Conv2D(filters1, (1, 1), strides=strides, name=conv_base + '2a')(input_tensor)
    x = BatchNormalization(axis=bn_axis, name=bn_base + '2a')(x)
    x = Activation('relu')(x)

    x = Conv2D(filters2, kernel_size, padding='same', name=conv_base + '2b')(x)
    x = BatchNormalization(axis=bn_axis, name=bn_base + '2b')(x)
    x = Activation('relu')(x)

    x = Conv2D(filters3, (1, 1), name=conv_base + '2c')(x)
    x = BatchNormalization(axis=bn_axis, name=bn_base + '2c')(x)

    shortcut = Conv2D(filters3, (1, 1), strides=strides, name=conv_base + '1')(input_tensor)
    shortcut = BatchNormalization(axis=bn_axis, name=bn_base + '1')(shortcut)

    x = layers.add([x, shortcut])
    x = Activation('relu')(x)
    return x

def ResNet50(include_top=True, weights='imagenet', input_shape=(224, 224, 3), pooling=None, classes=1000):
    img_input = Input(shape=input_shape)
    bn_axis = 3 if K.image_data_format() == 'channels_last' else 1

    x = ZeroPadding2D((3, 3))(img_input)
    x = Conv2D(64, (7, 7), strides=(2, 2), name='conv1')(x)
    x = BatchNormalization(axis=bn_axis, name='bn_conv1')(x)
    x = Activation('relu')(x)
    x = MaxPooling2D((3, 3), strides=(2, 2))(x)

    x = conv_block(x, 3, [64, 64, 256], stage=2, block='a', strides=(1, 1))
    x = identity_block(x, 3, [64, 64, 256], stage=2, block='b')
    x = identity_block(x, 3, [64, 64, 256], stage=2, block='c')

    x = conv_block(x, 3, [128, 128, 512], stage=3, block='a')
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='b')
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='c')
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='d')

    x = conv_block(x, 3, [256, 256, 1024], stage=4, block='a')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='b')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='c')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='d')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='e')
    x = identity_block(x, 3, [256, 256, 1024], stage=4, block='f')

    x = conv_block(x, 3, [512, 512, 2048], stage=5, block='a')
    x = identity_block(x, 3, [512, 512, 2048], stage=5, block='b')
    x = identity_block(x, 3, [512, 512, 2048], stage=5, block='c')

    x = AveragePooling2D((7, 7), name='avg_pool')(x)

    if include_top:
        x = Flatten()(x)
        x = Dense(classes, activation='softmax', name='fc1000')(x)
    else:
        if pooling == 'avg':
            x = GlobalAveragePooling2D()(x)
        elif pooling == 'max':
            x = GlobalMaxPooling2D()(x)

    model = Model(img_input, x, name='resnet50')

    # Load weights
    if weights == 'imagenet':
        weights_path = get_file(
            'resnet50_weights_tf_dim_ordering_tf_kernels.h5' if include_top else 'resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5',
            WEIGHTS_PATH if include_top else WEIGHTS_PATH_NO_TOP,
            cache_subdir='models',
            md5_hash='a7b3fe01876f51b976af0dea6bc144eb' if include_top else 'a268eb855778b3df3c7506639542a6af'
        )
        model.load_weights(weights_path)

    return model

# Run prediction
if __name__ == '__main__':
    model = ResNet50(include_top=True, weights='imagenet')

    img_path = '/Users/tanmaysingh/Desktop/Unknown.jpeg'  # Ensure this image exists in your working directory
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"{img_path} not found.")

    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    print('Input image shape:', x.shape)

    preds = model.predict(x)
    print('Predicted:', decode_predictions(preds, top=3)[0])  # Correct prediction print


# In[ ]:







Input image shape: (1, 224, 224, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 588ms/step
Predicted: [('n02058221', 'albatross', 0.68134284), ('n02051845', 'pelican', 0.19861573), ('n02033041', 'dowitcher', 0.0394922)]
